# EXTRACCIÓN ÁREAS CONSTRUIDAS #

In [1]:
import os
import sys
import logging
import geopandas as gpd
import fiona
from shapely.validation import make_valid

# -----------------------------------------------------------------------------
# 1. CONFIGURACIÓN DE LOGGING Y PARÁMETROS DE ENTRADA
# -----------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)

# Rutas principales
GDB_INPUT_PATH = r"D:\Maestria_Geomatica\Colombia en mapas\Cartografia vectorial\Carto100000_Colombia_DI_2022.gdb"
GDB_OUTPUT_DIR = r"D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP"
GDB_OUTPUT_NAME = "Areas_construidas_AOI.gdb"
GDB_OUTPUT_PATH = os.path.join(GDB_OUTPUT_DIR, GDB_OUTPUT_NAME)

# Reemplaza esta ruta por la de tu archivo .shp real
AOI_PATH = r"D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP\AOI_25_09_19_EPSG9377.shp"

# EPSG Objetivo: MAGNA-SIRGAS / Origen Nacional
TARGET_EPSG = 9377

# Datasets/Filtros requeridos
TARGET_DATASETS = [
    "Edificacion_ObraCivil",
    "Entidades_Territoriales_y_Unidades_Administrativas",
    "Instalaciones_Construcciones_Para_Transporte",
    "Transporte_Aereo"
]

In [2]:
# -----------------------------------------------------------------------------
# 2. FUNCIONES AUXILIARES
# -----------------------------------------------------------------------------
def load_and_prep_aoi(aoi_path: str, target_crs: int) -> gpd.GeoDataFrame:
    """Carga, valida geometrías y reproyecta el área de interés (AOI)."""
    logging.info(f"Cargando Área de Interés (AOI) desde: {aoi_path}")
    
    if not os.path.exists(aoi_path):
        raise FileNotFoundError(f"No se encontró el archivo de AOI en: {aoi_path}")

    aoi_gdf = gpd.read_file(aoi_path)

    # Corregir geometrías no válidas
    aoi_gdf["geometry"] = aoi_gdf["geometry"].apply(lambda geom: make_valid(geom) if geom else None)
    
    # Reproyectar a EPSG:9377 si difiere
    if aoi_gdf.crs is None or aoi_gdf.crs.to_epsg() != target_crs:
        logging.info(f"Reproyectando AOI a EPSG:{target_crs}...")
        aoi_gdf = aoi_gdf.to_crs(epsg=target_crs)
        
    return aoi_gdf

def get_layers_in_gdb(gdb_path: str) -> list:
    """Lista todas las capas disponibles en la File Geodatabase de entrada."""
    if not os.path.exists(gdb_path):
        raise FileNotFoundError(f"La GDB de origen no existe en: {gdb_path}")
    
    layers = fiona.listlayers(gdb_path)
    logging.info(f"Total de capas encontradas en la GDB de origen: {len(layers)}")
    return layers

# -----------------------------------------------------------------------------
# 3. FUNCIÓN PRINCIPAL DE PROCESAMIENTO
# -----------------------------------------------------------------------------
def main():
    logging.info("=== INICIANDO PROCESO DE CLIP GEOMÁTICO ===")
    
    # 1. Cargar AOI
    try:
        aoi_gdf = load_and_prep_aoi(AOI_PATH, TARGET_EPSG)
    except Exception as e:
        logging.error(f"Error al procesar el AOI: {e}")
        return

    # 2. Obtener todas las capas de la GDB de origen
    try:
        all_layers = get_layers_in_gdb(GDB_INPUT_PATH)
    except Exception as e:
        logging.error(f"Error al leer la GDB de origen: {e}")
        return

    # Palabras clave a excluir si no deseas procesar ciertas capas masivas (opcional)
    EXCLUDE_KEYWORDS = ["Relieve", "Toponimos"]
    
    # Filtrar capas de la lista 'all_layers' recién leída dentro de main()
    target_layers = [
        layer for layer in all_layers 
        if not any(excl.lower() in layer.lower() for excl in EXCLUDE_KEYWORDS)
    ]

    logging.info(f"Capas a evaluar para el recorte: {len(target_layers)}")

    # Crear directorio de salida si no existe
    os.makedirs(GDB_OUTPUT_DIR, exist_ok=True)

    processed_count = 0

    # 3. Iterar, recortar y guardar cada capa dentro de la nueva GDB
    for layer_name in target_layers:
        try:
            # Cargar la capa desde la GDB origen
            layer_gdf = gpd.read_file(GDB_INPUT_PATH, layer=layer_name)

            if layer_gdf.empty:
                continue

            # Reproyectar a EPSG:9377 si difiere
            if layer_gdf.crs is None or layer_gdf.crs.to_epsg() != TARGET_EPSG:
                layer_gdf = layer_gdf.to_crs(epsg=TARGET_EPSG)

            # Sanear geometrías
            layer_gdf["geometry"] = layer_gdf["geometry"].apply(lambda geom: make_valid(geom) if geom else None)

            # Aplicar Recorte (Clip)
            clipped_gdf = gpd.clip(layer_gdf, aoi_gdf)

            # Si no hay entidades dentro del AOI, saltar la exportación de la capa
            if clipped_gdf.empty:
                continue

            logging.info(f" -> Capa [{layer_name}]: {len(clipped_gdf)} elementos recortados.")

            clean_layer_name = layer_name.replace("/", "_").replace("\\", "_")

            # Exportar a la nueva GDB de salida
            clipped_gdf.to_file(
                GDB_OUTPUT_PATH,
                layer=clean_layer_name,
                driver="OpenFileGDB",
                engine="pyogrio"
            )
            processed_count += 1

        except Exception as e:
            logging.error(f"Error procesando la capa {layer_name}: {e}")

    logging.info("\n==================================================")
    logging.info(f"PROCESO FINALIZADO: {processed_count} capas guardadas en {GDB_OUTPUT_PATH}")
    logging.info("==================================================")

# 4. Ejecución
main()

2026-09-23 14:54:18,901 [INFO] === INICIANDO PROCESO DE CLIP GEOMÁTICO ===
2026-09-23 14:54:18,902 [INFO] Cargando Área de Interés (AOI) desde: D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP\AOI_25_09_19_EPSG9377.shp
2026-09-23 14:54:30,501 [INFO] Total de capas encontradas en la GDB de origen: 163
2026-09-23 14:54:30,502 [INFO] Capas a evaluar para el recorte: 163


C:\Users\Equipo\anaconda3\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


2026-09-23 14:58:02,340 [INFO]  -> Capa [Bosque]: 28 elementos recortados.
2026-09-23 14:58:02,970 [ERROR] Error procesando la capa Bosque: Unsupported geometry type
2026-09-23 15:02:44,839 [INFO]  -> Capa [Construccion_P]: 6998 elementos recortados.
2026-09-23 15:02:45,193 [INFO] Created 6,998 records
2026-09-23 15:03:40,913 [INFO]  -> Capa [Construccion_P_Anot]: 1595 elementos recortados.
2026-09-23 15:03:41,048 [INFO] Created 1,595 records
2026-09-23 15:04:30,356 [INFO]  -> Capa [Administrativo_P_Departamental]: 42 elementos recortados.
2026-09-23 15:04:30,433 [INFO] Created 42 records
2026-09-23 15:04:44,651 [INFO]  -> Capa [Administrativo_P_Anot]: 478 elementos recortados.
2026-09-23 15:04:44,721 [INFO] Created 478 records
2026-09-23 15:04:58,346 [INFO]  -> Capa [Administrativo_P]: 470 elementos recortados.
2026-09-23 15:04:58,530 [INFO] Created 470 records
2026-09-23 15:05:09,529 [INFO]  -> Capa [Administrativo_R_Anot]: 45 elementos recortados.
2026-09-23 15:05:09,685 [INFO] Crea